# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² human protein dataset using the `mlcroissant` library, directly from its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# The Dataset.metadata object exposes all schema contents
print(f"Dataset Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published on: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced via their `@id` in the schema.

Let's enumerate and inspect record sets, then list their fields and columns.

In [ ]:
# List all record sets available in this dataset
record_sets = dataset.metadata.record_sets

print(f"Number of record sets: {len(record_sets)}\n")
for idx, rs in enumerate(record_sets):
    print(f"Record Set {idx+1}: @id: {rs.id}")
    print(f"    Name: {rs.name}")
    print(f"    Description: {rs.description}")
    print(f"    Fields (@id): {[field.id for field in rs.fields]}")
    print(f"    Columns (@id): {[col.id for col in rs.columns] if hasattr(rs, 'columns') else 'N/A'}")
    print("")

For illustration, let's print the first few records from each record set using their `@id`.

In [ ]:
# Iterate and print a few records from each record set, referencing all by @id
for rs in record_sets:
    print(f"Sample from Record Set '@id': {rs.id} ({rs.name})")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        print(record)
        if i == 2:
            break
    print("\n---\n")

## 3. Data Extraction
Load data from all record sets into pandas DataFrames for analysis. Use record set and field `@id`s as keys.

In [ ]:
# Prepare DataFrames for each record set, using @id as dictionary keys
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for Record Set '@id': {record_set_id} ({len(df)} records)")

# For demonstration, display the columns and first rows of the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print("\nField (@id) columns in first record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All operations reference columns by their `@id`.

In [ ]:
# Suppose we analyze the first main record set
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Working with Record Set: {main_record_set_id}")

    # Identify a numeric field by @id (domain knowledge: commonly MW or coverage are relevant)
    # For demonstration, let's try 'cr:MW' for Molecular Weight, if present
    candidate_numeric_fields = ['cr:MW', 'cr:Coverage', 'cr:Peptide_count', 'cr:Unique_peptide_count']
    numeric_field_id = None
    for field in candidate_numeric_fields:
        if field in df.columns:
            numeric_field_id = field
            break
    if numeric_field_id is not None:
        print(f"Using numeric field '@id': {numeric_field_id}")

        # Set a threshold
        threshold = df[numeric_field_id].quantile(0.75)

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (top quartile):\n")
        display(filtered_df.head())

        # Normalize the field (z-score)
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' (z-score):\n")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categorical field, such as 'cr:description' if present
        group_candidate_fields = ['cr:description', 'cr:Sample']
        group_field_id = None
        for field in group_candidate_fields:
            if field in filtered_df.columns:
                group_field_id = field
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped filtered data by '{group_field_id}':\n")
            display(grouped_df.head())
    else:
        print("No obvious numeric field found in this record set. Please inspect columns:")
        print(df.columns.tolist())
else:
    print("No main record set found in metadata.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All fields referenced by their `@id`.

Below, we plot the distribution of the selected numeric field, and a grouped bar plot if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id is not None:
    # Distribution plot
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {main_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Grouped barplot if we have a group field
    if 'group_field_id' in locals() and group_field_id:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10, 6))
        sns.barplot(y=group_means.index, x=group_means.values)
        plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.show()


## 6. Conclusion
In this notebook, we:
- Loaded the FAIR² dataset metadata and records referencing all elements by their Croissant `@id`.
- Reviewed dataset structure including record set and field identifiers.
- Loaded all record sets into DataFrames, inspected columns, and analyzed records.
- Demonstrated filtering, normalization, grouping, and visualization based on field `@id`s.

This workflow can be adapted for any Croissant-formatted dataset with `mlcroissant` for FAIR and reproducible data science.